In [1]:
# =====================================================
# NYC Jobs Pipeline – Bronze → Silver → Gold
# Fully Optimized with Source Data Analysis, Medallion Architecture, Layer Tests, Feature Engineering and Visualization
# =====================================================

In [28]:
# Import pyspark

import findspark
findspark.init()

In [29]:
# Import libraries

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.types import *
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [30]:
# =====================================================
# Initialize Spark Session with Performance Optimizations
# - Enable Adaptive Query Execution for performance
# - Tune shuffle partitions
# =====================================================

In [31]:
spark = SparkSession.builder \
    .appName("NYC_Jobs_Medallion") \
    .config("spark.sql.adaptive.enabled", True) \
    .config("spark.sql.adaptive.coalescePartitions.enabled", True) \
    .config("spark.sql.shuffle.partitions", 200) \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [32]:
# =====================================================
# Define Medallion Layer Paths
# =====================================================

In [33]:
gold_path   = "/notebook/nyc_jobs_gold"

In [34]:
# =====================================================
# Bronze Layer – Raw CSV ingestion + minimal test
# =====================================================

In [35]:
'''
csv_path = "/dataset/nyc-jobs.csv"

# Read CSV
df_bronze = spark.read.option("header", True).csv(csv_path)

# Optional: cache if reused
df_bronze.cache()

# Validate
row_count = df_bronze.count()
col_count = len(df_bronze.columns)

if row_count == 0 or col_count == 0:
    raise ValueError("Bronze layer ingestion failed.")

print("Bronze ingestion complete, rows:", row_count, "columns:", col_count)'''

'\ncsv_path = "/dataset/nyc-jobs.csv"\n\n# Read CSV\ndf_bronze = spark.read.option("header", True).csv(csv_path)\n\n# Optional: cache if reused\ndf_bronze.cache()\n\n# Validate\nrow_count = df_bronze.count()\ncol_count = len(df_bronze.columns)\n\nif row_count == 0 or col_count == 0:\n    raise ValueError("Bronze layer ingestion failed.")\n\nprint("Bronze ingestion complete, rows:", row_count, "columns:", col_count)'

In [ ]:

df = spark.read.csv("/dataset/nyc-jobs.csv", header=True)
df.printSchema()

In [ ]:
# =====================================================
# Source Data Analysis – Column types, nulls, categorical distributions
# =====================================================

In [ ]:
def source_data_analysis(df):
    print("==== Column Data Types ====")
    df.printSchema()
    
    numeric_cols = ["Job ID", "# Of Positions", "Level", "Salary Range From", "Salary Range To"]
    print("\n==== Summary Statistics (Numerical Columns) ====")
    df.select([col(c).cast("double") for c in numeric_cols]).describe().show()
    
    print("\n==== Count of Missing / Null Values per Column ====")
    for c in df.columns:
        null_count = df.filter(col(c).isNull() | (col(c)=="")).count()
        print(f"{c}: {null_count}")
    
    categorical_cols = ["Agency", "Job Category", "Full-Time/Part-Time indicator", "Business Title"]
    print("\n==== Top 5 Unique Values for Categorical Columns ====")
    for c in categorical_cols:
        print(f"\nColumn: {c}")
        df.groupBy(c).count().orderBy(desc("count")).show(5)

source_data_analysis(df_bronze)

In [ ]:
# =====================================================
# Silver Layer – Cleaning, Feature Engineering, and Tests
# =====================================================

In [ ]:
def silver_layer_enhanced(df):
    df = df.dropDuplicates()
    
    # Drop heavy/unnecessary columns
    drop_cols = ["Job Description", "Minimum Qual Requirements", "Additional Information",
                 "To Apply", "Recruitment Contact"]
    df = df.drop(*drop_cols)
    
    # Type casting
    df = df.withColumn("Job ID", col("Job ID").cast("int")) \
           .withColumn("# Of Positions", col("# Of Positions").cast("int")) \
           .withColumn("Salary Range From", col("Salary Range From").cast("double")) \
           .withColumn("Salary Range To", col("Salary Range To").cast("double")) \
           .withColumn("Posting Date", to_date("Posting Date")) \
           .withColumn("Process Date", to_date("Process Date")) \
           .withColumn("Level", col("Level").cast("int"))
    
    # ===== Feature Engineering =====
    # Salary normalization & gap
    df = df.withColumn("Salary_Factor",
                       when(col("Salary Frequency")=="Hourly", 2080)
                       .when(col("Salary Frequency")=="Daily", 260)
                       .otherwise(1))
    df = df.withColumn("Salary_From_Annual", col("Salary Range From")*col("Salary_Factor")) \
           .withColumn("Salary_To_Annual", col("Salary Range To")*col("Salary_Factor")) \
           .withColumn("Avg_Salary", (col("Salary_From_Annual")+col("Salary_To_Annual"))/2) \
           .withColumn("Salary_Gap", col("Salary_To_Annual")-col("Salary_From_Annual"))
    
    # Degree extraction & encoding
    df = df.withColumn("Degree_Level",
                       when(col("Minimum Qual Requirements").rlike("PhD|Doctorate"), "PhD")
                       .when(col("Minimum Qual Requirements").rlike("Master|MBA|MS|MSc"), "Masters")
                       .when(col("Minimum Qual Requirements").rlike("baccalaureate|Bachelor"), "Bachelors")
                       .otherwise("Other")) \
           .withColumn("Degree_Encoded",
                       when(col("Degree_Level")=="PhD", 3)
                       .when(col("Degree_Level")=="Masters", 2)
                       .when(col("Degree_Level")=="Bachelors", 1)
                       .otherwise(0))
    
    # Full-Time / Part-Time encoding
    df = df.withColumn("FT_PT_Encoded",
                       when(col("Full-Time/Part-Time indicator")=="F", 1)
                       .when(col("Full-Time/Part-Time indicator")=="P", 0)
                       .otherwise(-1))
    
    # Level-to-positions ratio
    df = df.withColumn("Level_to_Positions_Ratio", col("Level") / (col("# Of Positions")+1))
    
    # Skill count
    df = df.withColumn("Skill_Count", size(split(regexp_replace(col("Preferred Skills"), "â€¢",""), "[,\\n\\.]")))
    
    # Agency + JobCategory
    df = df.withColumn("Agency_JobCategory", concat_ws("_", col("Agency"), col("Job Category")))
    
    # Posting month & year
    df = df.withColumn("Posting_Month", month("Posting Date")) \
           .withColumn("Posting_Year", year("Posting Date"))
    
    # Silver layer validation
    errors = []
    if df.groupBy("Job ID").count().filter(col("count")>1).count() > 0:
        errors.append("Duplicate Job IDs found.")
    if df.filter(col("Avg_Salary").isNull()).count() > 0:
        errors.append("Null Avg_Salary found.")
    if df.filter(col("Salary_Gap")<0).count() > 0:
        errors.append("Negative Salary_Gap found.")
    if df.filter(~col("Degree_Encoded").isin([0,1,2,3])).count() > 0:
        errors.append("Invalid Degree_Encoded values.")
    if df.filter(col("Posting Date").isNull()).count() > 0:
        errors.append("Null Posting Date found.")
    if errors:
        raise ValueError("Silver layer validation failed:\n" + "\n".join(errors))
    else:
        print("Silver layer validation successfully completed.")
    
    return df

df_silver = silver_layer_enhanced(df_bronze)

In [ ]:
# =====================================================
# Gold Layer – KPI Computation, Validation, and Storage
# =====================================================

In [ ]:
def gold_layer(df):
    # Top 10 Job Categories
    kpi1 = df.groupBy("Job Category").count().orderBy(desc("count")).limit(10)
    
    # Salary distribution per category
    kpi2 = df.groupBy("Job Category").agg(
        avg("Avg_Salary").alias("Avg_Salary"),
        min("Avg_Salary").alias("Min_Salary"),
        max("Avg_Salary").alias("Max_Salary"),
        stddev("Avg_Salary").alias("StdDev")
    )
    
    # Highest salary per agency
    windowSpec = Window.partitionBy("Agency").orderBy(desc("Salary_To_Annual"))
    kpi3 = df.withColumn("rank", row_number().over(windowSpec)) \
             .filter(col("rank")==1) \
             .select("Agency","Business Title","Salary_To_Annual")
    
    # Average salary last 2 years
    max_date = df.select(max("Posting Date")).collect()[0][0]
    cutoff_date = pd.to_datetime(max_date) - pd.Timedelta(days=730)
    df_recent = df.filter(col("Posting Date") >= cutoff_date)
    kpi4 = df_recent.groupBy("Agency").agg(avg("Avg_Salary").alias("Avg_Salary_Last_2_Years"))
    
    # Highest paid skills
    df_skills = df.withColumn("Skill", explode(split(regexp_replace(col("Preferred Skills"), "â€¢",""), "[,\\n\\.]"))) \
                  .withColumn("Skill", trim(col("Skill"))) \
                  .filter((length(col("Skill"))>=4) & (length(col("Skill"))<=80))
    skill_freq = df_skills.groupBy("Skill").count().filter(col("count")>=5)
    kpi5 = df_skills.join(skill_freq, "Skill") \
                    .groupBy("Skill") \
                    .agg(avg("Avg_Salary").alias("Avg_Salary")) \
                    .orderBy(desc("Avg_Salary"))
    
    # Degree vs Avg Salary
    kpi6 = df.groupBy("Degree_Level").agg(avg("Avg_Salary").alias("Avg_Salary")) \
             .orderBy(desc("Avg_Salary"))
    
    # Gold layer tests
    errors = []
    if kpi1.count() != 10:
        errors.append("Top 10 Job Categories KPI does not have 10 rows.")
    if kpi4.count() == 0:
        errors.append("Avg_Salary_Last_2_Years KPI empty.")
    if kpi5.filter(col("Avg_Salary").isNotNull()).count() == 0:
        errors.append("No valid skills found in KPI5.")
    if kpi6.count() == 0:
        errors.append("Degree vs Salary KPI empty.")
    if errors:
        raise ValueError("Gold layer validation failed:\n" + "\n".join(errors))
    else:
        print("Gold layer validation passed.")
    
    # Save Gold KPIs separately
    kpi1.write.mode("overwrite").parquet(f"{gold_path}/top10_jobs")
    kpi2.write.mode("overwrite").parquet(f"{gold_path}/salary_dist")
    kpi3.write.mode("overwrite").parquet(f"{gold_path}/highest_salary_per_agency")
    kpi4.write.mode("overwrite").parquet(f"{gold_path}/avg_salary_last_2_years")
    kpi5.write.mode("overwrite").parquet(f"{gold_path}/highest_paid_skills")
    kpi6.write.mode("overwrite").parquet(f"{gold_path}/degree_salary_corr")
    
    # Optional: Save all KPIs in a single file (union)
    from functools import reduce
    all_kpis = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), 
                      [kpi1, kpi2, kpi3, kpi4, kpi5, kpi6])
    all_kpis.write.mode("overwrite").parquet(f"{gold_path}/all_kpis_combined")
    
    return kpi1, kpi2, kpi3, kpi4, kpi5, kpi6

kpi1, kpi2, kpi3, kpi4, kpi5, kpi6 = gold_layer(df_silver)

In [ ]:
# =====================================================
# Visualizations (Pandas + Matplotlib/Seaborn)
# =====================================================

In [ ]:
# KPI1 – Top 10 Job Categories
kpi1_pd = kpi1.toPandas()
plt.figure(figsize=(10,6))
sns.barplot(data=kpi1_pd, y="Job Category", x="count", palette="viridis")
plt.title("Top 10 Job Categories")
plt.xlabel("Number of Job Postings")
plt.ylabel("Job Category")
plt.show()

# KPI2 – Salary Distribution per Job Category
kpi2_pd = kpi2.toPandas()
plt.figure(figsize=(12,6))
sns.boxplot(data=kpi2_pd, x="Job Category", y="Avg_Salary", palette="coolwarm")
plt.xticks(rotation=45)
plt.title("Salary Distribution per Job Category")
plt.ylabel("Average Salary (Annual)")
plt.xlabel("Job Category")
plt.show()

# KPI3 – Highest Salary per Agency
kpi3_pd = kpi3.toPandas()
plt.figure(figsize=(12,6))
sns.barplot(data=kpi3_pd, x="Agency", y="Salary_To_Annual", palette="magma")
plt.xticks(rotation=90)
plt.title("Highest Salary per Agency")
plt.ylabel("Salary (Annual)")
plt.xlabel("Agency")
plt.show()

# KPI4 – Average Salary Last 2 Years per Agency
kpi4_pd = kpi4.toPandas()
plt.figure(figsize=(12,6))
sns.barplot(data=kpi4_pd, x="Agency", y="Avg_Salary_Last_2_Years", palette="plasma")
plt.xticks(rotation=90)
plt.title("Average Salary in Last 2 Years per Agency")
plt.ylabel("Average Salary")
plt.xlabel("Agency")
plt.show()

# KPI5 – Highest Paid Skills
kpi5_pd = kpi5.limit(20).toPandas()  # Top 20 skills
plt.figure(figsize=(10,6))
sns.barplot(data=kpi5_pd, y="Skill", x="Avg_Salary", palette="cividis")
plt.title("Top 20 Highest Paid Skills")
plt.xlabel("Average Salary (Annual)")
plt.ylabel("Skill")
plt.show()

# KPI6 – Degree vs Avg Salary
kpi6_pd = kpi6.toPandas()
plt.figure(figsize=(8,5))
sns.barplot(data=kpi6_pd, x="Degree_Level", y="Avg_Salary", order=["Other","Bachelors","Masters","PhD"], palette="Set2")
plt.title("Correlation: Higher Degree vs Average Salary")
plt.xlabel("Degree Level")
plt.ylabel("Average Salary (Annual)")
plt.show()

print("Pipeline complete – Bronze → Source Analysis → Silver → Gold with updated feature engineering, layer tests, all 6 KPIs, and visualizations")